## 面试问题

完成判定：模型自称完成 vs 客观后置验证，怎么防假完成？

## 回答主线

模型会假完成：声称做完但目标没达成。正确做法是客观后置验证——用 goal predicate 检查权威状态回读，而非采信模型自称。本 Notebook 让退款 agent 在第 1 步过早自称完成（订单仍 paid），对比只信模型（假完成被放行、停在 paid）与客观验证（拒绝假完成、继续到真退款 refunded）。

## 真实案例

退款 agent：第 0 步 verify，第 1 步 say_done（自称完成但没退款），第 2 步才真正 refund。goal predicate 是 `order_status == refunded`。数据为教学状态，不代表真实系统。

In [1]:
def refund_agent(state):  # 一个会过早自称完成的退款 agent。
    if state["step"] == 0:  # 第 0 步先校验金额。
        return {"action": "verify", "claims_done": False}  # 校验但未声称完成。
    if state["step"] == 1:  # 第 1 步过早声称完成。
        return {"action": "say_done", "claims_done": True}  # 声称完成但还没真退款。
    return {"action": "refund", "claims_done": True}  # 第 2 步真正执行退款并声称完成。

initial = {"step": 0, "order_status": "paid"}  # 初始订单未退款。
print("初始订单状态:", initial["order_status"])  # 展示初始状态。
for step in range(3):  # 预览 agent 三步的提议。
    print("  第", step, "步提议:", refund_agent({"step": step, "order_status": "paid"}))  # 展示每步动作与自称。

初始订单状态: paid
  第 0 步提议: {'action': 'verify', 'claims_done': False}
  第 1 步提议: {'action': 'say_done', 'claims_done': True}
  第 2 步提议: {'action': 'refund', 'claims_done': True}


## 基线（Baseline）

反面基线：只信模型自称完成。第 1 步 agent 声称完成，循环立即停止，但订单状态仍是 `paid`——假完成被放行。

In [2]:
def apply_action(state, proposal):  # 根据动作更新订单状态。
    new_state = dict(state)  # 拷贝状态。
    if proposal["action"] == "refund":  # 只有 refund 真正改变订单。
        new_state["order_status"] = "refunded"  # 退款成功。
    new_state["step"] = state["step"] + 1  # 推进步数。
    return new_state  # 返回新状态。

def run_trust_model(agent, state, max_steps=5):  # 只信模型自称完成的循环。
    for _ in range(max_steps):  # 逐步执行。
        proposal = agent(state)  # agent 提议。
        state = apply_action(state, proposal)  # 应用动作。
        if proposal["claims_done"]:  # 模型自称完成就停。
            return {"stopped_at_step": state["step"], "order_status": state["order_status"], "reason": "model_claim"}  # 采信自称停止。
    return {"stopped_at_step": state["step"], "order_status": state["order_status"], "reason": "max_steps"}  # 兜底。

trust_result = run_trust_model(refund_agent, initial)  # 运行只信模型的循环。
print("只信模型结果:", trust_result)  # 展示在订单仍为 paid 时就假完成停止。

只信模型结果: {'stopped_at_step': 2, 'order_status': 'paid', 'reason': 'model_claim'}


## 失败案例与修正

只信模型在 `paid` 时就停（假完成）。修正是客观验证：模型自称完成时还要 goal predicate 校验权威状态回读，不满足就拒绝 finish、继续循环，直到订单真的 refunded。

In [3]:
def goal_reached(state):  # 客观完成条件：订单必须真实为已退款。
    return state["order_status"] == "refunded"  # 以权威状态回读为准。

def run_verified(agent, state, max_steps=5):  # 客观验证完成的循环。
    for _ in range(max_steps):  # 逐步执行。
        proposal = agent(state)  # agent 提议。
        state = apply_action(state, proposal)  # 应用动作。
        if proposal["claims_done"] and goal_reached(state):  # 自称完成且客观达成才停。
            return {"stopped_at_step": state["step"], "order_status": state["order_status"], "reason": "verified_done"}  # 客观完成停止。
    return {"stopped_at_step": state["step"], "order_status": state["order_status"], "reason": "max_steps"}  # 兜底。

verified_result = run_verified(refund_agent, initial)  # 运行客观验证的循环。
print("客观验证结果:", verified_result)  # 展示拒绝假完成继续到真退款。

客观验证结果: {'stopped_at_step': 3, 'order_status': 'refunded', 'reason': 'verified_done'}


In [4]:
print("只信模型停在状态:", trust_result["order_status"], "reason", trust_result["reason"])  # 展示假完成仍为 paid。
print("客观验证停在状态:", verified_result["order_status"], "reason", verified_result["reason"])  # 展示真完成 refunded。
print("停止步数 只信模型 vs 客观验证:", trust_result["stopped_at_step"], verified_result["stopped_at_step"])  # 对比停止步数。

只信模型停在状态: paid reason model_claim
客观验证停在状态: refunded reason verified_done
停止步数 只信模型 vs 客观验证: 2 3


In [5]:
is_fake_trust = trust_result["order_status"] != "refunded"  # 只信模型是否为假完成。
is_fake_verified = verified_result["order_status"] != "refunded"  # 客观验证是否为假完成。
print("只信模型是假完成:", is_fake_trust)  # 展示只信模型放行了假完成。
print("客观验证是假完成:", is_fake_verified)  # 展示客观验证杜绝了假完成。

只信模型是假完成: True
客观验证是假完成: False


## 结果解读

只信模型在第 1 步 agent 自称完成时就停，订单仍是 paid（假完成）；客观验证拒绝这次假完成，循环继续到第 2 步真正退款、状态变 refunded 才停。要点：完成是客观事实，用权威回读的 goal predicate 判定，拒绝假完成后要有继续或升级路径。

In [6]:
assert trust_result["order_status"] == "paid"  # 只信模型在未退款时就停止即假完成。
assert trust_result["reason"] == "model_claim"  # 只信模型因自称而停。
assert verified_result["order_status"] == "refunded"  # 客观验证停在真实退款状态。
assert verified_result["reason"] == "verified_done"  # 客观验证因目标达成而停。
assert verified_result["stopped_at_step"] > trust_result["stopped_at_step"]  # 客观验证多跑了必要的退款步。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
